In [ ]:
print("Début exécution")

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from scipy.stats import linregress

from pykrige.ok import OrdinaryKriging

import ipywidgets as widgets
from ipywidgets import (
    IntSlider, FloatSlider, VBox, HBox, HTML, interactive_output
)
from IPython.display import display

In [ ]:
print("Import OK")

In [2]:
# ----------------------------
# 1) Champ synthétique
# ----------------------------
def spherical_covariance(h, sill=1.0, rng=25.0, nugget=0.0):
    """
    Covariance sphérique correspondant à un variogramme sphérique :
    gamma(h) = nugget + sill * [1.5(h/a) - 0.5(h/a)^3] pour 0 < h < a
             = nugget + sill                             pour h >= a

    Ici on renvoie la covariance C(h) = (sill + nugget) - gamma(h)
    avec C(0) = sill + nugget.
    """
    h = np.asarray(h, dtype=float)
    c = np.zeros_like(h, dtype=float)

    # partie structurée
    hr = h / rng
    inside = h < rng
    c[inside] = sill * (1 - 1.5 * hr[inside] + 0.5 * hr[inside] ** 3)
    c[~inside] = 0.0

    # effet pépite au point
    c[h == 0] += nugget

    return c


def simulate_gaussian_field(
    x_block_centers, y_block_centers,
    sill=1.0, rng=25.0, nugget=0.0,
    seed=0
):
    """
    Simule un champ gaussien sur les centres de blocs à partir d'une covariance sphérique.
    """
    rng_np = np.random.default_rng(seed)

    coords = np.column_stack([x_block_centers, y_block_centers])
    d = cdist(coords, coords)

    cov = spherical_covariance(d, sill=sill, rng=rng, nugget=nugget)

    # petite régularisation numérique
    cov = cov + 1e-10 * np.eye(cov.shape[0])

    z = rng_np.multivariate_normal(
        mean=np.zeros(len(coords)),
        cov=cov
    )
    return z


# ----------------------------
# 2) Plus proche voisin
# ----------------------------
def nearest_neighbor_grid(x_obs, y_obs, z_obs, xg, yg):
    """
    Interpolation type polygones d'influence / plus proche voisin sur une grille.
    """
    grid_points = np.column_stack([xg.ravel(), yg.ravel()])
    obs_points = np.column_stack([x_obs, y_obs])

    d = cdist(grid_points, obs_points)
    idx = np.argmin(d, axis=1)

    return z_obs[idx].reshape(xg.shape)


def nearest_neighbor_points(x_obs, y_obs, z_obs, xq, yq):
    """
    Valeur plus proche voisin en des points quelconques.
    """
    q = np.column_stack([xq, yq])
    obs = np.column_stack([x_obs, y_obs])

    d = cdist(q, obs)
    idx = np.argmin(d, axis=1)
    return z_obs[idx]


# ----------------------------
# 3) Krigeage ordinaire
# ----------------------------
def ordinary_kriging_grid(
    x_obs, y_obs, z_obs,
    gridx, gridy,
    variogram_range=25.0,
    sill=1.0,
    nugget=0.0,
    nlags=10
):
    """
    Krigeage ordinaire sur grille avec variogramme sphérique imposé.
    """
    OK = OrdinaryKriging(
        x_obs,
        y_obs,
        z_obs,
        variogram_model="spherical",
        variogram_parameters={
            "sill": float(sill),
            "range": float(variogram_range),
            "nugget": float(nugget),
        },
        nlags=nlags,
        verbose=False,
        enable_plotting=False,
        coordinates_type="euclidean"
    )

    z_krig, ss_krig = OK.execute("grid", gridx, gridy)
    return np.asarray(z_krig), np.asarray(ss_krig)

In [6]:
print("UI construite")

UI construite


In [3]:
def compare_kriging_vs_pi(
    n_points=12,
    seed=0,
    noise_std=0.15,
    block_size=8,
    domain_size=80,
    variogram_range=25.0,
    sill=1.0,
    nugget=0.0,
    nlags=10,
    n_scatter_points=1500
):
    # ----------------------------
    # Domaine et blocs
    # ----------------------------
    nx = domain_size // block_size
    ny = domain_size // block_size

    if nx < 2 or ny < 2:
        raise ValueError("Taille de blocs trop grande pour le domaine.")

    x_edges = np.arange(0, domain_size + block_size, block_size)
    y_edges = np.arange(0, domain_size + block_size, block_size)

    x_centers = x_edges[:-1] + block_size / 2
    y_centers = y_edges[:-1] + block_size / 2

    xg, yg = np.meshgrid(x_centers, y_centers)

    # ----------------------------
    # Champ synthétique "vrai"
    # ----------------------------
    z_true_flat = simulate_gaussian_field(
        xg.ravel(),
        yg.ravel(),
        sill=sill,
        rng=variogram_range,
        nugget=nugget,
        seed=seed
    )
    z_true = z_true_flat.reshape(yg.shape)

    # ----------------------------
    # Echantillons
    # ----------------------------
    rng_np = np.random.default_rng(seed + 12345)

    x_obs = rng_np.uniform(0, domain_size, size=n_points)
    y_obs = rng_np.uniform(0, domain_size, size=n_points)

    # "vraie" valeur du bloc le plus proche pour chaque point d'échantillonnage
    ix = np.clip(np.floor(x_obs / block_size).astype(int), 0, nx - 1)
    iy = np.clip(np.floor(y_obs / block_size).astype(int), 0, ny - 1)

    z_obs_true = z_true[iy, ix]
    z_obs = z_obs_true + rng_np.normal(0, noise_std, size=n_points)

    # ----------------------------
    # Interpolations
    # ----------------------------
    z_nn = nearest_neighbor_grid(x_obs, y_obs, z_obs, xg, yg)

    z_ok, ss_ok = ordinary_kriging_grid(
        x_obs, y_obs, z_obs,
        gridx=x_centers,
        gridy=y_centers,
        variogram_range=variogram_range,
        sill=sill,
        nugget=nugget,
        nlags=nlags
    )

    # ----------------------------
    # Z* en fonction de Z
    # Ici on compare sur les blocs du domaine
    # ----------------------------
    z_true_vec = z_true.ravel()
    z_nn_vec = z_nn.ravel()
    z_ok_vec = z_ok.ravel()

    # sous-échantillonnage pour lisibilité du nuage
    n_all = len(z_true_vec)
    n_use = min(n_scatter_points, n_all)
    idx = rng_np.choice(n_all, size=n_use, replace=False)

    z_true_s = z_true_vec[idx]
    z_nn_s = z_nn_vec[idx]
    z_ok_s = z_ok_vec[idx]

    reg_nn = linregress(z_true_s, z_nn_s)
    reg_ok = linregress(z_true_s, z_ok_s)

    # ----------------------------
    # Histogrammes
    # ----------------------------
    global_min = min(z_true.min(), z_nn.min(), z_ok.min())
    global_max = max(z_true.max(), z_nn.max(), z_ok.max())
    bins = np.linspace(global_min, global_max, 18)

    # ----------------------------
    # Figure
    # ----------------------------
    fig, axes = plt.subplots(
        3, 3,
        figsize=(16, 14),
        constrained_layout=True
    )

    cmap = "viridis"
    vmin = global_min
    vmax = global_max

    # ===== Ligne 1 =====
    # Champ vrai
    im0 = axes[0, 0].imshow(
        z_true,
        origin="lower",
        extent=[0, domain_size, 0, domain_size],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest"
    )
    axes[0, 0].scatter(x_obs, y_obs, c="white", edgecolor="black", s=40, zorder=3)
    axes[0, 0].set_title("Champ synthétique (vrai)")
    axes[0, 0].set_xlabel("X")
    axes[0, 0].set_ylabel("Y")

    # Krigeage
    im1 = axes[0, 1].imshow(
        z_ok,
        origin="lower",
        extent=[0, domain_size, 0, domain_size],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest"
    )
    axes[0, 1].scatter(x_obs, y_obs, c="white", edgecolor="black", s=40, zorder=3)
    axes[0, 1].set_title("Interpolation par krigeage ordinaire")
    axes[0, 1].set_xlabel("X")
    axes[0, 1].set_ylabel("Y")

    # Plus proche voisin
    im2 = axes[0, 2].imshow(
        z_nn,
        origin="lower",
        extent=[0, domain_size, 0, domain_size],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest"
    )
    axes[0, 2].scatter(x_obs, y_obs, c="white", edgecolor="black", s=40, zorder=3)
    axes[0, 2].set_title("Interpolation plus proche voisin")
    axes[0, 2].set_xlabel("X")
    axes[0, 2].set_ylabel("Y")

    # grille de blocs en gris clair
    for ax in axes[0, :]:
        for xe in x_edges:
            ax.axvline(x=xe, color="lightgray", linewidth=0.8, alpha=0.8, zorder=2)
        for ye in y_edges:
            ax.axhline(y=ye, color="lightgray", linewidth=0.8, alpha=0.8, zorder=2)

    cbar = fig.colorbar(im0, ax=axes[0, :], shrink=0.9)
    cbar.set_label("Valeur")

    # ===== Ligne 2 =====
    # nuage krigeage
    ax = axes[1, 0]
    ax.scatter(z_true_s, z_ok_s, s=14, alpha=0.35)
    xx = np.linspace(min(z_true_s.min(), z_ok_s.min()), max(z_true_s.max(), z_ok_s.max()), 200)
    ax.plot(xx, xx, linestyle="--", linewidth=2, label="y = x")
    ax.plot(xx, reg_ok.slope * xx + reg_ok.intercept, linewidth=2,
            label=f"Régr. OK : y={reg_ok.slope:.2f}x+{reg_ok.intercept:.2f}")
    ax.set_title("Krigeage : Z* en fonction de Z")
    ax.set_xlabel("Z vrai")
    ax.set_ylabel("Z estimé")
    ax.legend(fontsize=9)

    # nuage NN
    ax = axes[1, 1]
    ax.scatter(z_true_s, z_nn_s, s=14, alpha=0.35)
    xx = np.linspace(min(z_true_s.min(), z_nn_s.min()), max(z_true_s.max(), z_nn_s.max()), 200)
    ax.plot(xx, xx, linestyle="--", linewidth=2, label="y = x")
    ax.plot(xx, reg_nn.slope * xx + reg_nn.intercept, linewidth=2,
            label=f"Régr. NN : y={reg_nn.slope:.2f}x+{reg_nn.intercept:.2f}")
    ax.set_title("Plus proche voisin : Z* en fonction de Z")
    ax.set_xlabel("Z vrai")
    ax.set_ylabel("Z estimé")
    ax.legend(fontsize=9)

    # comparaison sur le même graphe
    ax = axes[1, 2]
    ax.scatter(z_true_s, z_ok_s, s=14, alpha=0.25, label="Krigeage")
    ax.scatter(z_true_s, z_nn_s, s=14, alpha=0.25, label="Plus proche voisin")
    xx = np.linspace(
        min(z_true_s.min(), z_ok_s.min(), z_nn_s.min()),
        max(z_true_s.max(), z_ok_s.max(), z_nn_s.max()),
        200
    )
    ax.plot(xx, xx, linestyle="--", linewidth=2, label="y = x")
    ax.plot(xx, reg_ok.slope * xx + reg_ok.intercept, linewidth=2,
            label=f"Régr. OK (pente={reg_ok.slope:.2f})")
    ax.plot(xx, reg_nn.slope * xx + reg_nn.intercept, linewidth=2,
            label=f"Régr. NN (pente={reg_nn.slope:.2f})")
    ax.set_title("Comparaison directe des nuages")
    ax.set_xlabel("Z vrai")
    ax.set_ylabel("Z estimé")
    ax.legend(fontsize=9)

    # ===== Ligne 3 =====
    ax = axes[2, 0]
    ax.hist(z_true_vec, bins=bins, alpha=0.7, density=True)
    ax.set_title("Histogramme - champ vrai")
    ax.set_xlabel("Valeur")
    ax.set_ylabel("Densité")

    ax = axes[2, 1]
    ax.hist(z_ok_vec, bins=bins, alpha=0.7, density=True)
    ax.set_title("Histogramme - krigeage")
    ax.set_xlabel("Valeur")
    ax.set_ylabel("Densité")

    ax = axes[2, 2]
    ax.hist(z_true_vec, bins=bins, alpha=0.35, density=True, label="Vrai")
    ax.hist(z_ok_vec, bins=bins, alpha=0.35, density=True, label="Krigeage")
    ax.hist(z_nn_vec, bins=bins, alpha=0.35, density=True, label="Plus proche voisin")
    ax.set_title("Histogrammes comparatifs")
    ax.set_xlabel("Valeur")
    ax.set_ylabel("Densité")
    ax.legend()

    # ----------------------------
    # résumé texte
    # ----------------------------
    txt = (
        f"Krigeage  : pente={reg_ok.slope:.3f}, intercept={reg_ok.intercept:.3f}, R²={reg_ok.rvalue**2:.3f}\n"
        f"PPV / PI  : pente={reg_nn.slope:.3f}, intercept={reg_nn.intercept:.3f}, R²={reg_nn.rvalue**2:.3f}\n"
        f"Bruit     : écart-type du bruit ajouté aux échantillons = {noise_std:.2f}\n"
        f"Blocs     : taille = {block_size}"
    )
    fig.suptitle("Comparaison krigeage vs polygones d'influence", fontsize=16, y=1.02)
    fig.text(0.01, -0.01, txt, fontsize=11, family="monospace")

    plt.show()

In [7]:
print("Fonction appelée")

Fonction appelée


In [4]:
title = HTML("<h3>Comparaison Krigeage vs Polygones d'influence</h3>")

w_n_points = IntSlider(
    value=12, min=4, max=60, step=1,
    description="Nb points", continuous_update=False
)

w_seed = IntSlider(
    value=0, min=0, max=100, step=1,
    description="Seed", continuous_update=False
)

w_noise = FloatSlider(
    value=0.15, min=0.0, max=1.0, step=0.01,
    description="Bruit", continuous_update=False
)

w_block = IntSlider(
    value=8, min=4, max=20, step=2,
    description="Taille bloc", continuous_update=False
)

w_range = FloatSlider(
    value=25.0, min=8.0, max=40.0, step=1.0,
    description="Range", continuous_update=False
)

w_sill = FloatSlider(
    value=1.0, min=0.2, max=3.0, step=0.1,
    description="Sill", continuous_update=False
)

w_nugget = FloatSlider(
    value=0.0, min=0.0, max=0.8, step=0.05,
    description="Nugget", continuous_update=False
)

w_nlags = IntSlider(
    value=10, min=4, max=20, step=1,
    description="nlags", continuous_update=False
)

out = interactive_output(
    compare_kriging_vs_pi,
    {
        "n_points": w_n_points,
        "seed": w_seed,
        "noise_std": w_noise,
        "block_size": w_block,
        "variogram_range": w_range,
        "sill": w_sill,
        "nugget": w_nugget,
        "nlags": w_nlags,
    }
)

ui = VBox([
    title,
    HBox([w_n_points, w_seed, w_noise, w_block]),
    HBox([w_range, w_sill, w_nugget, w_nlags]),
    out
])

display(ui)

In [5]:

import numpy as np
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from scipy.stats import linregress
from pykrige.ok import OrdinaryKriging

import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, VBox, HBox, HTML, interactive_output
from IPython.display import display


# ============================================================
# Utilitaires géostat
# ============================================================

def spherical_covariance(h, sill=1.0, rng=25.0, nugget=0.0):
    """
    Covariance sphérique correspondant à un variogramme sphérique.
    C(0) = sill + nugget
    C(h) = sill * (1 - 1.5 h/a + 0.5 (h/a)^3) pour 0 < h < a
         = 0 au-delà de la portée a
    """
    h = np.asarray(h, dtype=float)
    c = np.zeros_like(h, dtype=float)

    inside = h < rng
    hr = np.zeros_like(h, dtype=float)
    hr[inside] = h[inside] / rng

    c[inside] = sill * (1 - 1.5 * hr[inside] + 0.5 * hr[inside] ** 3)
    c[h == 0] += nugget
    return c


def simulate_gaussian_field(x, y, sill=1.0, rng=25.0, nugget=0.0, seed=0):
    """
    Simulation d'un champ gaussien sur une maille régulière.
    """
    coords = np.column_stack([x, y])
    d = cdist(coords, coords)
    cov = spherical_covariance(d, sill=sill, rng=rng, nugget=nugget)
    cov = cov + 1e-10 * np.eye(cov.shape[0])  # stabilité numérique

    rng_np = np.random.default_rng(seed)
    z = rng_np.multivariate_normal(np.zeros(len(coords)), cov)
    return z


def nearest_neighbor_grid(x_obs, y_obs, z_obs, xg, yg):
    """
    Interpolation polygones d'influence / plus proche voisin sur grille régulière.
    """
    obs = np.column_stack([x_obs, y_obs])
    q = np.column_stack([xg.ravel(), yg.ravel()])
    d = cdist(q, obs)
    idx = np.argmin(d, axis=1)
    return z_obs[idx].reshape(xg.shape)


def kriging_grid(x_obs, y_obs, z_obs, x_centers, y_centers, var_range, sill=1.0, nugget=0.0, nlags=10):
    """
    Krigeage ordinaire avec variogramme sphérique imposé = "bien ajusté".
    """
    ok = OrdinaryKriging(
        x_obs,
        y_obs,
        z_obs,
        variogram_model="spherical",
        variogram_parameters={
            "sill": float(sill),
            "range": float(var_range),
            "nugget": float(nugget),
        },
        nlags=nlags,
        verbose=False,
        enable_plotting=False,
        coordinates_type="euclidean",
    )
    z_ok, ss_ok = ok.execute("grid", x_centers, y_centers)
    return np.asarray(z_ok), np.asarray(ss_ok)


def build_regular_sampling_grid(domain_size, n_points_target):
    """
    Construit un échantillonnage régulier.
    On approxime le nombre de points demandé par une grille n_side x n_side.
    """
    n_side = max(2, int(np.round(np.sqrt(n_points_target))))
    xs = np.linspace(domain_size / (2 * n_side), domain_size - domain_size / (2 * n_side), n_side)
    ys = np.linspace(domain_size / (2 * n_side), domain_size - domain_size / (2 * n_side), n_side)
    x_obs, y_obs = np.meshgrid(xs, ys)
    return x_obs.ravel(), y_obs.ravel(), n_side


# ============================================================
# Fonction principale
# ============================================================

def compare_methods(
    var_range=25.0,
    n_points_target=16,
    seed=0,
    noise_std=0.15,
    block_size=8,
    domain_size=80,
    sill=1.0,
    nugget=0.0,
    nlags=10,
):
    """
    bruit = écart-type du bruit gaussien ajouté aux valeurs échantillonnées.
    Il représente ici une mesure imparfaite / variabilité locale / erreur simple.
    """
    # ----------------------------
    # Maille régulière du modèle
    # ----------------------------
    nx = domain_size // block_size
    ny = domain_size // block_size
    if nx < 3 or ny < 3:
        raise ValueError("Augmenter le domaine ou diminuer la taille des blocs.")

    x_edges = np.arange(0, domain_size + block_size, block_size)
    y_edges = np.arange(0, domain_size + block_size, block_size)
    x_centers = x_edges[:-1] + block_size / 2
    y_centers = y_edges[:-1] + block_size / 2
    xg, yg = np.meshgrid(x_centers, y_centers)

    # ----------------------------
    # Champ synthétique "vrai"
    # ----------------------------
    z_true = simulate_gaussian_field(
        xg.ravel(), yg.ravel(),
        sill=sill, rng=var_range, nugget=nugget, seed=seed
    ).reshape(yg.shape)

    # ----------------------------
    # Echantillonnage régulier
    # ----------------------------
    x_obs, y_obs, n_side = build_regular_sampling_grid(domain_size, n_points_target)

    # On associe à chaque point régulier la valeur du bloc correspondant
    ix = np.clip(np.floor(x_obs / block_size).astype(int), 0, nx - 1)
    iy = np.clip(np.floor(y_obs / block_size).astype(int), 0, ny - 1)

    rng_np = np.random.default_rng(seed + 1000)
    z_obs_true = z_true[iy, ix]
    z_obs = z_obs_true + rng_np.normal(0, noise_std, size=len(z_obs_true))

    # ----------------------------
    # Interpolations
    # ----------------------------
    z_ok, ss_ok = kriging_grid(
        x_obs, y_obs, z_obs,
        x_centers, y_centers,
        var_range=var_range, sill=sill, nugget=nugget, nlags=nlags
    )

    z_pi = nearest_neighbor_grid(x_obs, y_obs, z_obs, xg, yg)

    # ----------------------------
    # Données pour diagnostics
    # ----------------------------
    z_true_vec = z_true.ravel()
    z_ok_vec = z_ok.ravel()
    z_pi_vec = z_pi.ravel()

    reg_ok = linregress(z_true_vec, z_ok_vec)
    reg_pi = linregress(z_true_vec, z_pi_vec)

    mae_ok = np.mean(np.abs(z_ok_vec - z_true_vec))
    mae_pi = np.mean(np.abs(z_pi_vec - z_true_vec))

    # ----------------------------
    # Figure multi-lignes
    # ----------------------------
    fig = plt.figure(figsize=(16, 10), constrained_layout=True)
    gs = fig.add_gridspec(2, 3, height_ratios=[1.15, 1.0])

    axes = np.empty((2, 3), dtype=object)
    axes[0, 0] = fig.add_subplot(gs[0, 0])
    axes[0, 1] = fig.add_subplot(gs[0, 1])
    axes[0, 2] = fig.add_subplot(gs[0, 2])
    axes[1, 0:2] = [fig.add_subplot(gs[1, 0:2]), None]
    axes[1, 2] = fig.add_subplot(gs[1, 2])

    cmap = "viridis"
    vmin = min(z_true.min(), z_ok.min(), z_pi.min())
    vmax = max(z_true.max(), z_ok.max(), z_pi.max())

    # ----- ligne 1 : cartes
    maps = [
        (z_true, "Champ synthétique (vrai)"),
        (z_ok, "Interpolation par krigeage"),
        (z_pi, "Interpolation polygones d'influence"),
    ]

    for ax, (zmat, title) in zip(axes[0, :], maps):
        im = ax.imshow(
            zmat,
            origin="lower",
            extent=[0, domain_size, 0, domain_size],
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            interpolation="nearest",
            aspect="equal",
        )
        ax.scatter(x_obs, y_obs, c="white", edgecolor="black", s=38, zorder=4)
        ax.set_title(title, fontsize=13)
        ax.set_xlabel("X")
        ax.set_ylabel("Y")

        # Limites des blocs en gris clair
        for xe in x_edges:
            ax.axvline(xe, color="lightgray", linewidth=0.8, alpha=0.85, zorder=3)
        for ye in y_edges:
            ax.axhline(ye, color="lightgray", linewidth=0.8, alpha=0.85, zorder=3)

    cbar = fig.colorbar(im, ax=axes[0, :], shrink=0.9)
    cbar.set_label("Valeur")

    # ----- ligne 2 gauche : Z* vs Z
    ax = axes[1, 0]
    ax.scatter(z_true_vec, z_ok_vec, s=16, alpha=0.35, label="Krigeage")
    ax.scatter(z_true_vec, z_pi_vec, s=16, alpha=0.25, label="Polygones d'influence")

    xy_min = min(z_true_vec.min(), z_ok_vec.min(), z_pi_vec.min())
    xy_max = max(z_true_vec.max(), z_ok_vec.max(), z_pi_vec.max())
    xx = np.linspace(xy_min, xy_max, 200)

    ax.plot(xx, xx, linestyle="--", linewidth=2, label="y = x")
    ax.plot(xx, reg_ok.slope * xx + reg_ok.intercept, linewidth=2.3,
            label=f"Régr. krigeage : y={reg_ok.slope:.2f}x+{reg_ok.intercept:.2f}")
    ax.plot(xx, reg_pi.slope * xx + reg_pi.intercept, linewidth=2.3,
            label=f"Régr. PI : y={reg_pi.slope:.2f}x+{reg_pi.intercept:.2f}")

    ax.set_title("Relation Z* en fonction de Z", fontsize=13)
    ax.set_xlabel("Z vrai")
    ax.set_ylabel("Z estimé")
    ax.legend(fontsize=9, loc="best")
    ax.grid(alpha=0.25)

    # ----- ligne 2 droite : histogrammes comparatifs
    ax = axes[1, 2]
    bins = np.linspace(xy_min, xy_max, 18)
    ax.hist(z_true_vec, bins=bins, density=True, alpha=0.35, label="Champ vrai")
    ax.hist(z_ok_vec, bins=bins, density=True, alpha=0.35, label="Krigeage")
    ax.hist(z_pi_vec, bins=bins, density=True, alpha=0.35, label="Polygones d'influence")
    ax.set_title("Histogrammes comparatifs", fontsize=13)
    ax.set_xlabel("Valeur")
    ax.set_ylabel("Densité")
    ax.legend(fontsize=9)

    # ----- résumé
    fig.suptitle("Comparaison krigeage vs polygones d'influence", fontsize=16)

    txt = (
        f"Range du phénomène = {var_range:.1f} | "
        f"Échantillonnage régulier ≈ {n_side}x{n_side} = {len(x_obs)} points | "
        f"Taille des blocs = {block_size}\n"
        f"Bruit = écart-type du bruit gaussien ajouté aux valeurs échantillonnées = {noise_std:.2f}\n"
        f"Krigeage : pente={reg_ok.slope:.3f}, intercept={reg_ok.intercept:.3f}, R²={reg_ok.rvalue**2:.3f}, MAE={mae_ok:.3f}\n"
        f"PI       : pente={reg_pi.slope:.3f}, intercept={reg_pi.intercept:.3f}, R²={reg_pi.rvalue**2:.3f}, MAE={mae_pi:.3f}"
    )
    fig.text(0.01, -0.02, txt, fontsize=10.5, family="monospace")

    plt.show()


# ============================================================
# Widgets
# ============================================================

title = HTML("<h3>Maille régulière — Krigeage vs polygones d'influence</h3>")
subtitle = HTML(
    "<i>Bruit</i> = écart-type du bruit gaussien ajouté aux valeurs échantillonnées "
    "(mesure imparfaite / variabilité locale simplifiée)."
)

w_range = FloatSlider(value=25.0, min=8.0, max=45.0, step=1.0, description="Range", continuous_update=False)
w_npts = IntSlider(value=16, min=4, max=81, step=1, description="Nb points", continuous_update=False)
w_seed = IntSlider(value=0, min=0, max=100, step=1, description="Seed", continuous_update=False)
w_noise = FloatSlider(value=0.15, min=0.0, max=1.0, step=0.01, description="Bruit", continuous_update=False)
w_block = IntSlider(value=8, min=4, max=20, step=2, description="Taille bloc", continuous_update=False)

out = interactive_output(
    compare_methods,
    {
        "var_range": w_range,
        "n_points_target": w_npts,
        "seed": w_seed,
        "noise_std": w_noise,
        "block_size": w_block,
    }
)

ui = VBox([
    title,
    subtitle,
    HBox([w_range, w_npts, w_seed]),
    HBox([w_noise, w_block]),
    out
])

display(ui)
